# MRC -> Point Cloud -> Chroma (Clean Workflow)

This notebook is organized as:
1. **One setup block**: imports + reusable function definitions.
2. **Section blocks**: define input variables, then call one function.


In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path
from typing import Any, Dict, Optional

# Must be set before importing Chroma-related modules on some macOS setups.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import numpy as np
import yaml

import mrc_pointcloud as mpc
import qc_orthogonal_plots as qc_plots

from run_mrc_chroma_shape import (
    ADAPTIVE_MASK_FOREGROUND_THRESHOLD,
    CHROMA_SAMPLE_DEFAULTS,
    SHAPE_CONDITIONER_DEFAULTS,
    _build_chroma_sample_kwargs,
    _build_shape_conditioner_kwargs,
    _merge_dict_defaults,
    _save_sampled_proteins,
)

def load_config(config_path: Path) -> Dict[str, Any]:
    with config_path.open('r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError(f'{config_path} must contain a YAML object at top level.')
    return cfg

def build_runtime(repo_root: Path, cfg: Dict[str, Any], overrides: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    p = dict(cfg)
    if overrides:
        p.update(overrides)

    input_mrc = Path(str(p['input_mrc']))
    if not input_mrc.is_absolute():
        input_mrc = (repo_root / input_mrc).resolve()

    output_dir = Path(str(p['output_dir']))
    if not output_dir.is_absolute():
        output_dir = (repo_root / output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    adaptive_pooling = str(p.get('adaptive_pooling', 'binary_max')).strip().lower()
    if adaptive_pooling not in ('binary_max', 'mean_fft'):
        raise ValueError('adaptive_pooling must be 'binary_max' or 'mean_fft'.')

    seed = int(p.get('seed', 0))
    shape_conditioner_cfg = _merge_dict_defaults(
        SHAPE_CONDITIONER_DEFAULTS,
        p.get('shape_conditioner') if isinstance(p.get('shape_conditioner'), dict) else None,
    )
    chroma_sample_cfg = _merge_dict_defaults(
        CHROMA_SAMPLE_DEFAULTS,
        p.get('chroma_sample') if isinstance(p.get('chroma_sample'), dict) else None,
    )

    return {
        'repo_root': repo_root,
        'input_mrc': input_mrc,
        'output_dir': output_dir,
        'threshold_raw': p.get('threshold'),
        'threshold_quantile': float(p.get('threshold_quantile', 0.995)),
        'use_adaptive_bin': bool(p.get('use_adaptive_bin', True)),
        'adaptive_pooling': adaptive_pooling,
        'max_voxels': int(p.get('max_voxels', 2000)),
        'max_bin_rounds': int(p.get('max_bin_rounds', 32)),
        'adaptive_subsample_seed': p.get('adaptive_subsample_seed', seed),
        'max_points_conditioner': p.get('max_points_conditioner'),
        'conditioner_subsample_seed': int(p.get('conditioner_subsample_seed', seed)),
        'seed': seed,
        'api_key': p.get('api_key'),
        'shape_conditioner_cfg': shape_conditioner_cfg,
        'chroma_sample_cfg': chroma_sample_cfg,
    }

def run_pointcloud_stage(rt: Dict[str, Any]) -> Dict[str, Any]:
    data, voxel_size, origin, header = mpc.read_mrc(str(rt['input_mrc']))
    stats = mpc.inspect_mrc(data, quantiles=(0.5, 0.9, 0.95, 0.99, rt['threshold_quantile']))

    threshold_raw = rt['threshold_raw']
    if threshold_raw is None:
        if rt['use_adaptive_bin']:
            warnings.warn('threshold is null: using threshold_quantile for scalar threshold.', stacklevel=1)
        threshold = float(np.quantile(data.astype(np.float64), rt['threshold_quantile']))
    else:
        threshold = float(threshold_raw)

    mask_data = (data >= threshold).astype(np.float32)

    adaptive_meta = None
    if rt['use_adaptive_bin']:
        points, adaptive_meta, final_density = mpc.mrc_to_point_cloud_adaptive_cap(
            data=mask_data,
            voxel_size=voxel_size,
            origin=origin,
            threshold=ADAPTIVE_MASK_FOREGROUND_THRESHOLD,
            max_voxels=rt['max_voxels'],
            max_rounds=rt['max_bin_rounds'],
            subsample_seed=None if rt['adaptive_subsample_seed'] is None else int(rt['adaptive_subsample_seed']),
            adaptive_pooling=rt['adaptive_pooling'],
        )
        final_voxel_size = tuple(float(v) for v in adaptive_meta['final_voxel_size_xyz'])
    else:
        points = mpc.mrc_to_point_cloud(
            data=mask_data,
            voxel_size=voxel_size,
            origin=origin,
            threshold=ADAPTIVE_MASK_FOREGROUND_THRESHOLD,
            max_points=None,
            random_seed=None,
        )
        final_density = mask_data
        final_voxel_size = (float(voxel_size[0]), float(voxel_size[1]), float(voxel_size[2]))

    names = {
        'input_density_png': 'module1_2_15A_input_density_orthogonal.png',
        'input_density_thr0_png': 'module1_2_15A_input_density_ge_threshold_else_zero_orthogonal.png',
        'input_mask_png': 'module1_2_15A_input_mask_orthogonal.png',
        'final_density_mrc': 'module1_2_15A_final_density.mrc',
        'final_density_png': 'module1_2_15A_final_density_orthogonal.png',
        'points_png': 'module1_2_15A_points_orthogonal.png',
        'overlay_final_png': 'module1_2_15A_overlay_points_on_final.png',
        'overlay_input_mask_png': 'module1_2_15A_overlay_points_on_input_mask.png',
        'points_npy': 'module1_2_15A_points.npy',
        'metrics_json': 'module1_2_15A_pointcloud_metrics.json',
    }
    paths = {k: rt['output_dir'] / v for k, v in names.items()}

    qc_plots.plot_input_density_orthogonal(data, voxel_size=voxel_size, origin=origin, out_path=paths['input_density_png'])
    qc_plots.plot_input_density_hard_threshold_orthogonal(
        data, threshold=threshold, voxel_size=voxel_size, origin=origin, out_path=paths['input_density_thr0_png']
    )
    qc_plots.plot_input_mask_orthogonal(mask_data, voxel_size=voxel_size, origin=origin, out_path=paths['input_mask_png'])

    mpc.write_mrc(str(paths['final_density_mrc']), final_density, voxel_size=final_voxel_size, origin=origin, overwrite=True)
    qc_plots.plot_final_density_orthogonal(final_density, voxel_size=final_voxel_size, origin=origin, out_path=paths['final_density_png'])
    qc_plots.plot_points_orthogonal_slabs(
        points,
        grid_shape_zyx=final_density.shape,
        voxel_size=final_voxel_size,
        origin=origin,
        out_path=paths['points_png'],
    )
    qc_plots.plot_overlay_points_on_final_volume(
        final_density, voxel_size=final_voxel_size, origin=origin, points=points, out_path=paths['overlay_final_png']
    )
    qc_plots.plot_overlay_points_on_input_mask(
        mask_data,
        voxel_size_input=voxel_size,
        origin=origin,
        points=points,
        out_path=paths['overlay_input_mask_png'],
        voxel_size_final_for_slab=final_voxel_size,
    )

    np.save(paths['points_npy'], points)
    metrics = {
        'input_mrc': str(rt['input_mrc']),
        'shape_zyx': list(data.shape),
        'voxel_size_xyz': list(voxel_size),
        'origin_xyz': list(origin),
        'threshold': threshold,
        'selected_points': int(points.shape[0]),
        'stats': stats,
        'adaptive_meta': adaptive_meta,
    }
    with paths['metrics_json'].open('w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, sort_keys=True, default=str)
        f.write('\n')

    return {
        'rt': rt,
        'data': data,
        'voxel_size': voxel_size,
        'origin': origin,
        'header': header,
        'stats': stats,
        'threshold': threshold,
        'mask_data': mask_data,
        'adaptive_meta': adaptive_meta,
        'points': points,
        'final_density': final_density,
        'final_voxel_size': final_voxel_size,
        'paths': paths,
    }

def load_points_for_chroma(points_npy: Path, max_points_conditioner: Any, conditioner_seed: int):
    points_loaded = np.load(points_npy)
    points_for_chroma = points_loaded
    if max_points_conditioner is not None:
        cap = int(max_points_conditioner)
        if cap <= 0:
            raise ValueError('max_points_conditioner must be positive when set.')
        if points_for_chroma.shape[0] > cap:
            rng = np.random.default_rng(seed=int(conditioner_seed))
            idx = rng.choice(points_for_chroma.shape[0], size=cap, replace=False)
            points_for_chroma = points_for_chroma[idx]
    return points_loaded, points_for_chroma

def run_chroma(points_for_chroma: np.ndarray, rt: Dict[str, Any], output_suffix: str = 'ipynb_example') -> Dict[str, Any]:
    import torch
    from chroma import Chroma, api, conditioners

    if rt['api_key']:
        api.register_key(str(rt['api_key']))

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    chroma = Chroma()

    sc_kw = _build_shape_conditioner_kwargs(rt['shape_conditioner_cfg'])
    conditioner = conditioners.ShapeConditioner(
        points_for_chroma,
        chroma.backbone_network.noise_schedule,
        **sc_kw,
    ).to(device)

    torch.manual_seed(rt['seed'])
    sample_kw = _build_chroma_sample_kwargs(rt['chroma_sample_cfg'], conditioner, device)
    sample_out = chroma.sample(**sample_kw)
    if sample_kw.get('full_output'):
        shaped_protein, _full_out_dict = sample_out
    else:
        shaped_protein = sample_out

    out_pdb = rt['output_dir'] / f'module1_2_15A_shape_design.{output_suffix}.pdb'
    out_cif = rt['output_dir'] / f'module1_2_15A_shape_design.{output_suffix}.cif'
    saved = _save_sampled_proteins(shaped_protein, out_pdb, out_cif)

    return {'device': device, 'saved': saved, 'sample_kw': sample_kw}


## Section A: Inputs for MRC -> point cloud stage

Set only these variables, then run the cell.

In [ ]:
REPO_ROOT = Path.cwd().resolve()
CONFIG_PATH = REPO_ROOT / 'configs' / 'mrc_chroma_shape.yaml'

# Optional overrides here (keep empty dict to fully use YAML values).
POINTCLOUD_OVERRIDES = {
    # 'input_mrc': 'projects/input/module1_2_15A.mrc',
    # 'output_dir': 'projects/output',
    # 'threshold': 0.00871,
}

cfg = load_config(CONFIG_PATH)
runtime = build_runtime(REPO_ROOT, cfg, overrides=POINTCLOUD_OVERRIDES)
print('CONFIG_PATH:', CONFIG_PATH)
print('input_mrc:', runtime['input_mrc'])
print('output_dir:', runtime['output_dir'])

## Section B: Run MRC -> QC figures + point cloud

Generates PNGs/MRC/NPY/metrics in `output_dir`.

In [ ]:
pc_result = run_pointcloud_stage(runtime)
print('points shape:', pc_result['points'].shape, 'dtype:', pc_result['points'].dtype)
for key, p in pc_result['paths'].items():
    print(f'{key}:', p.resolve())

## Section C: Preview QC images

In [ ]:
from IPython.display import Image, display

preview_keys = [
    'input_density_png',
    'input_mask_png',
    'points_png',
    'overlay_final_png',
]
for k in preview_keys:
    display(Image(filename=str(pc_result['paths'][k])))

## Section D: Load `.npy` for downstream Chroma

In [ ]:
points_npy_path = pc_result['paths']['points_npy']
MAX_POINTS_CONDITIONER = runtime['max_points_conditioner']
CONDITIONER_SUBSAMPLE_SEED = runtime['conditioner_subsample_seed']

points_loaded, points_for_chroma = load_points_for_chroma(
    points_npy_path,
    MAX_POINTS_CONDITIONER,
    CONDITIONER_SUBSAMPLE_SEED,
)
print('loaded:', points_loaded.shape)
print('for_chroma:', points_for_chroma.shape)

## Section E: Chroma sampling

Set `RUN_CHROMA = True` when ready.

In [ ]:
RUN_CHROMA = False
OUTPUT_SUFFIX = 'ipynb_example'

if RUN_CHROMA:
    chroma_result = run_chroma(points_for_chroma, runtime, output_suffix=OUTPUT_SUFFIX)
    print('device:', chroma_result['device'])
    print('saved:', chroma_result['saved'])
else:
    print('Chroma skipped. Set RUN_CHROMA=True to run sampling.')